# 25/25Hz＋5fps：显示写端批量复制候选
保留前版；本次只把所有者共享显示写端改为批量复制。25/25Hz＋5fps、原电路、心率与运动算法、原1ms所有者线程间隔及全部保护不变；PS线程间隔仍保持原值。
前两单元不启动硬件，最下面单元需手动确认、放稳手指后仅运行一次。700点预热另计、正式60秒；失败保存，不重试。其它任务须先结束；屏幕独立供电并已上电。收尾时不要中断内核。


In [ ]:
from pathlib import Path
import sys
sys.dont_write_bytecode = True
from IPython.display import display, FileLink
assert sys.platform == 'linux', '仅在板上运行'
root = Path.cwd()
assert root.name == 'v3b25 owner display60', '请在新的候选目录运行'
from owner_schedule_guard import verify
checked = verify(root)
print({k: checked[k] for k in ('bundle_validated','source_members_unchanged','package_members','manifest_sha256')})
from run_owner_schedule60 import short, prepare, run
short_report = short(expected_manifest_sha256=checked['manifest_sha256'])
if (root/'lean-ps-short-results.zip').is_file():
    display(FileLink('lean-ps-short-results.zip'))
assert short_report['passed'], '无硬件短检查未通过，保留结果，不启动硬件'
ps_identity = short_report['ready_info']['owner_schedule_ps']
assert ps_identity['manifest_sha256'] == checked['manifest_sha256']
assert ps_identity['manifest_verified_before_original'] and not ps_identity['switch_changed_in_ps']
print('原700点READY秒数：', short_report['ready_info']['warmup_seconds'])
print('PS保持原线程切换间隔：', ps_identity['switch_seconds'])


In [ ]:
# 只准备浏览器波形，不启动采集。
prepared = prepare(expected_manifest_sha256=checked['manifest_sha256'])


In [ ]:
assert not globals().get('_owner_schedule_used', False), '已尝试，保留结果，不重跑'
confirmation = input('其它任务已结束，放稳手指；允许加载原电路并做一次60秒联合实测？输入 RUN-OWNER-DISPLAY60: ')
assert confirmation == 'RUN-OWNER-DISPLAY60', '未确认，不启动硬件'
_owner_schedule_used = True
print('700点预热另计、正式60秒；原保护不变，失败保存，不自动重试。', flush=True)
try:
    report = run(hardware=True, confirmation=confirmation, expected_manifest_sha256=checked['manifest_sha256'])
finally:
    if (root/'owner-display-results.zip').is_file():
        display(FileLink('owner-display-results.zip'))
    else:
        print('收尾/导出未确认，保留输出，不启动另一入口。')
